In [2]:
from PIL import Image
import requests
import re
import os
from transformers import CLIPProcessor, CLIPModel
import glob
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["A temporal aerial collage photo with ephemeral gully formed", "A temporal aerial collage photo with no ephemeral gully formed"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/opt/conda/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The

In [22]:

def read_images_for_tile(tile_number, directory):
    images = []
    for i in range(6):
        # Construct the filename based on the tile number and image sequence
        filename = f"neg_rgb_{i}_tile_{tile_number}.jpg"
        filepath = os.path.join(directory, filename)
        
        # Check if the file exists and load the image
        if os.path.exists(filepath):
            image = Image.open(filepath)
            images.append(image)
        else:
            print(f"File {filename} does not exist.")
    
    return images

# Specify the tile number and directory containing images
tile_number = 100
directory = '/root/home/data_jpg/'  # Replace with the path to your images

# Read images for the specified tile number
images = read_images_for_tile(tile_number, directory)

def get_unique_tiles(directory):
    tile_numbers = set()
    # Regular expression to match the tile number in the filename
    pattern = r'_(\d+)\.jpg'  # Matches the tile number before .jpg

    for filename in os.listdir(directory):
        match = re.search(pattern, filename)
        if match:
            tile_number = match.group(1)
            tile_numbers.add(tile_number)

    return sorted(tile_numbers)

In [23]:
tiles = get_unique_tiles(directory)

In [21]:
inputs = processor(text=["an aerial photo of a location with an ephemeral gully formed", "an aerial photo of a location with no ephemeral gully formed"], images=images, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities
print(probs)
# Determine if any image has a high probability for the gully class
gully_detected = (probs[:, 0] > 0.5).any().item()  # Check if any image has gully class probability > 0.5

if gully_detected:
    print("At least one image has an ephemeral gully.")
else:
    print("No ephemeral gully detected in any of the images.")

tile_number = 100
directory = '/root/home/data_jpg/'  # Replace with the path to your images

# Read images for the specified tile number
images = read_images_for_tile(tile_number, directory)

tensor([[0.2854, 0.7146],
        [0.3014, 0.6986],
        [0.1831, 0.8169],
        [0.1297, 0.8703],
        [0.2861, 0.7139],
        [0.1992, 0.8008]], grad_fn=<SoftmaxBackward0>)
No ephemeral gully detected in any of the images.


In [19]:
probs

tensor([[0.8889, 0.1111],
        [0.9538, 0.0462],
        [0.8749, 0.1251],
        [0.7142, 0.2858],
        [0.9312, 0.0688],
        [0.8492, 0.1508]], grad_fn=<SoftmaxBackward0>)

In [4]:

data_dir = '/root/home/data/'


# List all .jpg files in the specified directory (non-recursive)
jpg_files = glob.glob(f"{data_dir}/*.jpg")
#print(jpg_files)


In [11]:
device = "cuda"
model.to(device)

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [43]:
import tqdm
import torch
import json

base_path = '/mnt/Software/WACV-2025-Workshop-ViGIR'
test_set = os.path.join(base_path,'Testing_Set.json')

with open(test_set, 'r') as file:
    data = json.load(file)
    
tiles = list(data.keys())
gt_dict = create_gt_dict(data, tiles)

i=0

clip_results = {}
gullies=0
n_files = len(tiles)


# A temporal aerial collage photo with an ephemeral gully formed
for tile in tiles:
    print(f"Processing -- {tile} -- {i}/{n_files}")
    img = read_collage_with_tile(tile, data_dir)
    inputs = processor(text=["A photo with an ephemeral gully", "A photo with no ephemeral gully"], images=img, return_tensors="pt", padding=True)
    inputs.to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
    probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities
    #print(probs)
    
    # Determine if any image has a high probability for the gully class
    gully_detected = (probs[:, 0] > 0.5).any().item()  # Check if any image has gully class probability > 0.5
    #print(gully_detected)
    if gully_detected :
        gullies+=1
        #print(f"Gully present: {gullies}")
        clip_results[tile]=1
    else :
        clip_results[tile]=0
    i+=1
    del inputs
    

Processing -- 415 -- 0/311
pos_collage_415.jpg
Processing -- 1020 -- 1/311
neg_collage_1020.jpg
Processing -- 105 -- 2/311
neg_collage_105.jpg
Processing -- 439 -- 3/311
pos_collage_439.jpg
Processing -- 914 -- 4/311
pos_collage_914.jpg
Processing -- 1099 -- 5/311
neg_collage_1099.jpg
Processing -- 1065 -- 6/311
neg_collage_1065.jpg
Processing -- 373 -- 7/311
pos_collage_373.jpg
Processing -- 166 -- 8/311
pos_collage_166.jpg
Processing -- 396 -- 9/311
pos_collage_396.jpg
Processing -- 837 -- 10/311
neg_collage_837.jpg
Processing -- 685 -- 11/311
pos_collage_685.jpg
Processing -- 226 -- 12/311
pos_collage_226.jpg
Processing -- 956 -- 13/311
pos_collage_956.jpg
Processing -- 70 -- 14/311
neg_collage_70.jpg
Processing -- 604 -- 15/311
pos_collage_604.jpg
Processing -- 1067 -- 16/311
neg_collage_1067.jpg
Processing -- 118 -- 17/311
neg_collage_118.jpg
Processing -- 774 -- 18/311
neg_collage_774.jpg
Processing -- 521 -- 19/311
neg_collage_521.jpg
Processing -- 910 -- 20/311
pos_collage_910.

Processing -- 377 -- 175/311
pos_collage_377.jpg
Processing -- 950 -- 176/311
pos_collage_950.jpg
Processing -- 454 -- 177/311
neg_collage_454.jpg
Processing -- 686 -- 178/311
pos_collage_686.jpg
Processing -- 569 -- 179/311
pos_collage_569.jpg
Processing -- 236 -- 180/311
pos_collage_236.jpg
Processing -- 1004 -- 181/311
neg_collage_1004.jpg
Processing -- 19 -- 182/311
pos_collage_19.jpg
Processing -- 207 -- 183/311
pos_collage_207.jpg
Processing -- 301 -- 184/311
neg_collage_301.jpg
Processing -- 819 -- 185/311
neg_collage_819.jpg
Processing -- 887 -- 186/311
pos_collage_887.jpg
Processing -- 666 -- 187/311
pos_collage_666.jpg
Processing -- 789 -- 188/311
neg_collage_789.jpg
Processing -- 867 -- 189/311
pos_collage_867.jpg
Processing -- 967 -- 190/311
pos_collage_967.jpg
Processing -- 752 -- 191/311
neg_collage_752.jpg
Processing -- 691 -- 192/311
pos_collage_691.jpg
Processing -- 474 -- 193/311
neg_collage_474.jpg
Processing -- 854 -- 194/311
pos_collage_854.jpg
Processing -- 237 --

In [44]:
gullies

224

In [40]:
def create_gt_dict(data_dict, tiles):
    gt_dict = {}
    for tile in tiles:
        image_data = data_dict[tile] # label, labelers
        label = image_data['label']
        if label == '4':
            gt_dict[tile] = 1
        else :
            gt_dict[tile] = 0
    return gt_dict

def read_collage_with_tile(tile_number, directory):
    images = []
    
    filename = f"pos_collage_{tile_number}.jpg"
    filepath = os.path.join(directory, filename)
        
    if os.path.exists(filepath):
        img = Image.open(filepath)
        print(filename)
        return img
    else:
        filename = f"neg_collage_{tile_number}.jpg"
        filepath = os.path.join(directory, filename)
        img = Image.open(filepath)
        print(filename)

        return img
        

In [45]:
clip_results

{'415': 1,
 '1020': 0,
 '105': 1,
 '439': 1,
 '914': 1,
 '1099': 1,
 '1065': 0,
 '373': 1,
 '166': 1,
 '396': 1,
 '837': 1,
 '685': 1,
 '226': 1,
 '956': 0,
 '70': 1,
 '604': 1,
 '1067': 1,
 '118': 1,
 '774': 0,
 '521': 1,
 '910': 1,
 '975': 1,
 '352': 1,
 '761': 1,
 '1007': 0,
 '428': 1,
 '1038': 1,
 '50': 1,
 '838': 1,
 '126': 1,
 '1078': 1,
 '944': 1,
 '532': 1,
 '1041': 0,
 '540': 1,
 '128': 1,
 '722': 1,
 '478': 1,
 '639': 1,
 '668': 1,
 '343': 1,
 '1021': 0,
 '552': 0,
 '848': 1,
 '74': 0,
 '520': 1,
 '1032': 1,
 '615': 0,
 '536': 1,
 '1117': 0,
 '646': 1,
 '390': 1,
 '923': 0,
 '194': 1,
 '216': 1,
 '99': 1,
 '372': 1,
 '857': 0,
 '335': 1,
 '505': 1,
 '972': 0,
 '727': 1,
 '1064': 1,
 '740': 0,
 '182': 0,
 '316': 1,
 '706': 0,
 '193': 1,
 '882': 1,
 '932': 1,
 '174': 1,
 '440': 1,
 '965': 1,
 '1140': 1,
 '273': 1,
 '878': 1,
 '826': 1,
 '596': 1,
 '1123': 1,
 '1031': 1,
 '942': 1,
 '239': 1,
 '1005': 0,
 '139': 1,
 '11': 1,
 '855': 0,
 '836': 1,
 '97': 1,
 '124': 1,
 '568': 1,


In [46]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
# Convert dictionary values to lists
y_true = list(gt_dict.values())
y_pred = list(clip_results.values())

# Calculate metrics
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)

# Print results
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Accuracy: {accuracy}")

Precision: 0.5982142857142857
Recall: 0.7570621468926554
F1 Score: 0.6683291770573566
Accuracy: 0.572347266881029
